In [16]:
!pip install requests beautifulsoup4

In [20]:
# Import required libraries
import os
import requests
import re
from bs4 import BeautifulSoup  # This is the BeautifulSoup library import

In [22]:
def get_page():
    global url
    url = input("Enter url of a medium article: ")
    
    if not re.match(r'https?://medium.com/', url):
        print('Please enter a valid website, or make sure it is a medium article')
        sys.exit(1)
    
    # Add headers to mimic a browser request
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    
    res = requests.get(url, headers=headers)
    res.raise_for_status()
    soup = BeautifulSoup(res.text, 'html.parser')
    return soup

In [8]:
def clean(text):
    # Dictionary of replacements
    rep = {"<br>": "\n", "<br/>": "\n", "<li>": "\n"}
    rep = dict((re.escape(k), v) for k, v in rep.items())
    pattern = re.compile("|".join(rep.keys()))
    text = pattern.sub(lambda m: rep[re.escape(m.group(0))], text)
    # Fix the escape sequence by using a raw string (r prefix)
    text = re.sub(r'\<(.*?)\>', '', text)
    return text

In [24]:
def collect_text(soup):
    text = f'url: {url}\n\n'
    
    # Find the article content - Medium articles typically have article text in 'p' tags
    para_text = soup.find_all('p', class_='pw-post-body-paragraph')  # This class is commonly used in Medium articles
    
    print("Extracting paragraphs...")
    for para in para_text:
        # Clean the text before adding it
        cleaned_text = clean(para.text)
        if cleaned_text.strip():  # Only add non-empty paragraphs
            text += f"{cleaned_text}\n\n"
    
    return text

In [12]:
def save_file(text):
    if not os.path.exists('./scraped_articles'):
        os.mkdir('./scraped_articles')
    name = url.split("/")[-1]
    print(name)
    fname = f'scraped_articles/{name}.txt'
    
    # Write the text to a file
    with open(fname, 'w', encoding='utf-8') as file:
        file.write(text)
    
    print(f'File saved in directory {fname}')

In [26]:
if __name__ == '__main__':
    text = collect_text(get_page())
    save_file(text)

Enter url of a medium article:  https://medium.com/@subashgandyer/papa-what-is-a-neural-network-c5e5cc427c7


Extracting paragraphs...
papa-what-is-a-neural-network-c5e5cc427c7
File saved in directory scraped_articles/papa-what-is-a-neural-network-c5e5cc427c7.txt
